# B2B Radar — Google Colab pipeline

This notebook installs a pinned checkout of the repository, downloads or copies the comments file from Google Drive, validates every JSONL record, and exposes guarded dry-run, smoke-run, full-run, and resume controls. It never starts expensive ML work unless the corresponding flag is explicitly enabled. Do not save outputs containing credentials or raw comments.

## 1. Select a GPU runtime

In Colab choose **Runtime → Change runtime type → GPU**, then edit the parameters below. Use a stable `RUN_ID`; the same value and dataset checksum are required for resume.

In [ ]:
REPOSITORY_URL = "https://github.com/osmirnov34/b2b-radar.git"
CODE_REF = "main"  # Prefer a reviewed commit SHA for reproducible runs.
PROJECT_DIR = "/content/b2b-radar"

DRIVE_FILE_ID = "1RreCEUjXYy1qSB0N66o0W6XuWVsD5xYN"
DRIVE_INPUT_PATH = ""  # Optional mounted-Drive path; takes precedence over DRIVE_FILE_ID.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/b2b-radar"
EXPECTED_RECORDS = 260_000
RECORD_COUNT_TOLERANCE = 0.25
RUN_ID = "colab-full-001"
MODEL_REVISION = "3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3"
EMBEDDING_BATCH_SIZE = 16

RUN_SMOKE = False
SMOKE_RECORDS = 2_000
RUN_FULL = False
RESUME_RUN_DIR = ""  # Example: /content/drive/MyDrive/b2b-radar/ml-runs/colab-full-001
RESTART_FROM = ""  # Empty or a stage name such as embeddings.
STOP_AFTER = ""  # Empty for all stages; use a stage name for a deliberate partial run.

## 2. Clone and install the maintained pipeline

A non-empty project directory is rejected instead of overwritten. Restart the Colab runtime for a clean checkout. The resolved Git commit is printed and must be retained with experiment records.

In [ ]:
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
    raise RuntimeError(f"Python {platform.python_version()} is unsupported; expected 3.11, 3.12, or 3.13")
project_root = Path(PROJECT_DIR)
if project_root.exists():
    raise FileExistsError(f"Project directory already exists; restart the runtime: {project_root}")
subprocess.run(
    ["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(project_root)],
    check=True,
)
subprocess.run(["git", "-C", str(project_root), "checkout", CODE_REF], check=True)
resolved_commit = subprocess.run(
    ["git", "-C", str(project_root), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
hnswlib_install = "binary-or-existing"
if sys.version_info[:2] == (3, 13):
    if shutil.which("g++") is None:
        raise RuntimeError("Python 3.13 requires g++ to build hnswlib; use the standard Colab runtime")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "hnswlib==0.8.0", "--no-binary=hnswlib"],
        check=True,
    )
    hnswlib_install = "controlled-source-build"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{project_root}[analysis]", "gdown>=5.2"],
    check=True,
)
os.chdir(project_root)
hdbscan = importlib.import_module("hdbscan")
hnswlib = importlib.import_module("hnswlib")
np = importlib.import_module("numpy")
sentence_transformers = importlib.import_module("sentence_transformers")
torch = importlib.import_module("torch")
umap = importlib.import_module("umap")

dependency_report = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "commit": resolved_commit,
    "project": str(project_root),
    "hnswlib_install": hnswlib_install,
    "sentence_transformers_import": sentence_transformers.__name__,
    "dependencies": {name: version(name) for name in (
        "numpy", "torch", "sentence-transformers", "umap-learn", "hdbscan", "hnswlib"
    )},
}
print(dependency_report)

## 3. Mount Drive and acquire the dataset

For a restricted shared file, add a shortcut to My Drive and set `DRIVE_INPUT_PATH`. Otherwise the public file ID is downloaded with `gdown`. The source is copied to Colab's local disk for faster reads; results remain under mounted Drive for resume.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_project = Path(DRIVE_PROJECT_DIR)
drive_project.mkdir(parents=True, exist_ok=True)
(drive_project / "colab-environment.json").write_text(
    json.dumps(dependency_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
local_dataset = project_root / "data/raw/comments.jsonl"
local_dataset.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
if DRIVE_INPUT_PATH:
    mounted_source = Path(DRIVE_INPUT_PATH)
    if not mounted_source.is_file():
        raise FileNotFoundError(f"Mounted Drive file not found: {mounted_source}")
    shutil.copy2(mounted_source, local_dataset)
else:
    import gdown

    downloaded = gdown.download(id=DRIVE_FILE_ID, output=str(local_dataset), quiet=False)
    if downloaded is None or not local_dataset.is_file():
        raise RuntimeError("Drive download failed; add the file to My Drive and set DRIVE_INPUT_PATH")
print({"dataset_path": str(local_dataset), "size_gib": round(local_dataset.stat().st_size / 1024**3, 3)})

## 4. Validate the complete input

Only JSONL matching `ExportedComment` is accepted. JSON arrays, CSV, archives, HTML permission pages, invalid UTF-8, excessive schema errors, or a large record-count mismatch stop here. Diagnostics contain hashes and field types, never comment text.

In [ ]:
from src.ml import DatasetFormat, detect_dataset_format, inspect_comments_jsonl

format_report = detect_dataset_format(local_dataset)
print(format_report.model_dump())
if format_report.detected != DatasetFormat.JSONL:
    raise RuntimeError(
        f"Expected JSONL, detected {format_report.detected.value}; do not convert without reviewing the source format"
    )
inspection = inspect_comments_jsonl(
    local_dataset,
    expected_records=EXPECTED_RECORDS,
    expected_records_tolerance=RECORD_COUNT_TOLERANCE,
    max_error_rate=0.01,
)
inspection_summary = {
    "sha256": inspection.sha256,
    "records": inspection.contract_valid,
    "invalid": inspection.json_invalid + inspection.contract_invalid,
    "error_rate": inspection.error_rate,
    "unique_videos": inspection.unique_videos,
    "unique_queries": inspection.unique_queries,
    "record_count_comparison": (
        inspection.record_count_comparison.model_dump()
        if inspection.record_count_comparison is not None
        else None
    ),
    "critical_errors": inspection.critical_errors,
}
print(inspection_summary)
if not inspection.is_usable:
    raise RuntimeError("Dataset validation is blocked; review aggregate critical_errors above")

## 5. Build an untracked Colab configuration

The tracked examples remain unchanged. The generated runtime config uses the current Python interpreter, local input, persistent Drive run storage, the validated record count, and strict final evaluation.

In [ ]:
from src.ml import EmbeddingConfig
from src.operations import (
    PipelineConfig,
    PipelineStage,
    dry_run_pipeline,
    render_dry_run_report,
    render_smoke_run_report,
    run_pipeline,
    run_smoke_pipeline,
)

base_config_path = project_root / "configs/pipeline.example.json"
base_config = PipelineConfig.model_validate_json(base_config_path.read_text(encoding="utf-8"))
base_embedding = EmbeddingConfig.model_validate_json(
    (project_root / "configs/embeddings.example.json").read_text(encoding="utf-8")
)
colab_embedding = base_embedding.model_copy(
    update={
        "model_revision": MODEL_REVISION,
        "device": "cuda",
        "batch_size": EMBEDDING_BATCH_SIZE,
        "threads": 2,
    }
)
colab_embedding_path = Path("/content/embeddings.colab.json")
colab_embedding_path.write_text(f"{colab_embedding.model_dump_json(indent=2)}\n", encoding="utf-8")
pipeline_config = base_config.model_copy(
    update={
        "source_dataset": local_dataset,
        "runs_root": drive_project / "ml-runs",
        "run_id": RUN_ID,
        "python_executable": Path(sys.executable),
        "expected_records": EXPECTED_RECORDS,
        "expected_records_tolerance": RECORD_COUNT_TOLERANCE,
        "minimum_free_gb": 10.0,
        "embeddings_config": colab_embedding_path,
    }
)
runtime_config_path = Path("/content/pipeline.colab.json")
runtime_config_path.write_text(f"{pipeline_config.model_dump_json(indent=2)}\n", encoding="utf-8")
os.environ.setdefault("HF_HOME", "/content/huggingface-cache")
print({"config": str(runtime_config_path), "runs_root": str(pipeline_config.runs_root)})

## 6. Inspect runtime resources and execute read-only dry-run

In [ ]:
disk = shutil.disk_usage("/content")
resource_summary = {
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gib": (
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        if torch.cuda.is_available()
        else None
    ),
    "local_free_disk_gib": round(disk.free / 1024**3, 2),
}
print(resource_summary)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected; select a GPU runtime before smoke/full execution")

rng = np.random.default_rng(42)
compatibility_vectors = rng.normal(size=(48, 12)).astype(np.float32)
reduced = umap.UMAP(n_components=3, n_neighbors=5, n_epochs=20, random_state=42).fit_transform(
    compatibility_vectors
)
labels = hdbscan.HDBSCAN(min_cluster_size=3).fit_predict(reduced)
index = hnswlib.Index(space="cosine", dim=compatibility_vectors.shape[1])
index.init_index(max_elements=len(compatibility_vectors), ef_construction=20, M=8)
index.add_items(compatibility_vectors, np.arange(len(compatibility_vectors)))
neighbors, _ = index.knn_query(compatibility_vectors[:2], k=3)
if reduced.shape != (48, 3) or labels.shape != (48,) or neighbors.shape != (2, 3):
    raise RuntimeError("ML dependency compatibility check returned unexpected shapes")
if not np.isfinite(reduced).all():
    raise RuntimeError("ML dependency compatibility check produced non-finite values")
print({"compatibility_check": "passed", "reduced_shape": reduced.shape, "clusters": len(set(labels))})

In [ ]:
resume_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else None
restart_stage = PipelineStage(RESTART_FROM) if RESTART_FROM else None
stop_stage = PipelineStage(STOP_AFTER) if STOP_AFTER else None
dry_report = dry_run_pipeline(
    pipeline_config,
    project_root,
    config_path=runtime_config_path,
    run_dir=resume_dir,
    resume=resume_dir is not None,
    restart_from=restart_stage,
    stop_after=stop_stage,
)
print(render_dry_run_report(dry_report))
if not dry_report.can_run:
    raise RuntimeError("Dry-run is BLOCKED; do not enable smoke or full execution")

## 7. Optional smoke-run

Return to the parameter cell and set `RUN_SMOKE = True` only after dry-run is allowed. This executes stages 1–11 on a deterministic whole-video sample, stores its workspace in Drive, and never evaluates or publishes it.

In [ ]:
smoke_report = None
if RUN_SMOKE:
    smoke_config = pipeline_config.model_copy(update={"run_id": f"{RUN_ID}-check"})
    smoke_report = run_smoke_pipeline(
        smoke_config,
        project_root,
        records=SMOKE_RECORDS,
        seed=42,
        config_path=runtime_config_path,
        echo_progress=True,
    )
    print(render_smoke_run_report(smoke_report))
    if not smoke_report.full_run_allowed:
        raise RuntimeError("Smoke-run did not pass; keep RUN_FULL disabled")
else:
    print("Smoke-run is disabled. Set RUN_SMOKE=True only after reviewing dry-run.")

## 8. Guarded full run or resume

Set `RUN_FULL = True` only after a successful smoke-run. A new run uses `RUN_ID`; resume requires the same input checksum and `RESUME_RUN_DIR`. Use `RESTART_FROM` only after the resume dry-run explicitly plans the replacement. The pipeline pauses for manual review before final export.

In [ ]:
run_manifest = None
if RUN_FULL:
    starting_new_run = resume_dir is None
    if starting_new_run and (not RUN_SMOKE or smoke_report is None or not smoke_report.full_run_allowed):
        raise RuntimeError("A successful smoke-run in this session is required before a new full run")
    run_manifest = run_pipeline(
        pipeline_config,
        project_root,
        run_dir=resume_dir,
        resume=resume_dir is not None,
        restart_from=restart_stage,
        stop_after=stop_stage,
        echo_progress=True,
    )
    print({"run_id": run_manifest.run_id, "status": run_manifest.status.value, "message": run_manifest.message})
else:
    print("Full execution is disabled. Keep it disabled until smoke-run passes.")

## 9. Manual-review handoff

The first complete automatic pass is expected to stop at manual evaluation. Keep `RUN_FULL` disabled while annotating the generated local review sample. Then update a copied evaluation config with `validation_completed=true`, set `manual_annotations`, re-run dry-run with `RESUME_RUN_DIR` and `RESTART_FROM="evaluation"`, and only then explicitly resume. Publication remains a separate reviewed CLI operation and is intentionally absent from this notebook.